In [ ]:
import sqlite3 #Creates the database and runs SQL queries.
import re #Finds and removes unwanted text using patterns.
import requests #Sends requests to the Groq AI API.
from getpass import getpass #getpass-Accepts the API key without showing it on the screen.

# 1. Create sample database
conn = sqlite3.connect(":memory:")#This creates a temporary database in RAM (memory). It disappears when the program ends.

conn.executescript("""
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    department_name TEXT
);

CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    salary REAL,
    department_id INTEGER
);

INSERT INTO departments VALUES
(1, 'IT'), (2, 'HR'), (3, 'Finance');

INSERT INTO employees VALUES
(1, 'Ravi', 25, 60000, 1),
(2, 'Priya', 28, 75000, 1),
(3, 'Amit', 30, 45000, 2),
(4, 'Sneha', 26, 55000, 3),
(5, 'Neha', 24, 40000, 2);
""")

conn.commit()

# 2. Enter Groq API key
api_key = getpass( #The API key allows your program to communicate with Groq AI.
    "Enter Groq API key (press Enter for demo mode): "
).strip() #strip()-removes extra spaces from the beginning and end.

API_URL = "https://api.groq.com/openai/v1/chat/completions" #API_URL-is the address where the request is sent.
# Updated model to a current active one
MODEL = "openai/gpt-oss-20b" #MODEL specifies which AI model should process the question.


# 3. Function to call Groq AI
def ask_groq(prompt): #This function sends your prompt to Groq.
    response = requests.post( #requests is a library used to communicate with websites and api response means it stores the response given by api
        
        API_URL,
        headers={
            "Authorization": f"Bearer {api_key}", #Authorization → Sends your API key to prove that you have access.Inserts your actual API key into the text.
            "Content-Type": "application/json" #
        },
        json={
            "model": MODEL,
            "messages": [
                {"role": "user", "content": prompt} #messages contains the instructions and question.
                #"role": "user" → Says that the message is from the user."content": prompt-it contains instruction or prompt
            ],
            "temperature": 0 #Controls how much variation the AI uses when generating its answer.0 → Encourages more consistent, predictable answers.
        },
        timeout=60 #Waits up to 60 seconds for the request to complete before timing out.if api require too mush time to generate response 
    )

    if not response.ok: #If the API request fails, the program reports the error.
        raise RuntimeError(
            f"Groq API error {response.status_code}: " #response.status_code → gives the error code, like 404 or 401
            f"{response.text}"
        )

    return response.json()["choices"][0]["message"]["content"] #This extracts the AI's reply, which should contain the SQL query.

# response.json()	Converts the API response into Python data.
# ["choices"]	Gets the list of AI response choices.
# [0]	Selects the first choice.
# ["message"]	Gets the message from that choice.
# ["content"]	Gets the actual text written by the AI.
# return	Sends that text back to the calling function.

# 4. Generate SQL using AI or demo rules
def generate_sql(question): #This function converts the user's English question into SQL.

    if api_key:
        prompt = f"""
You are an SQLite SQL assistant.

Database schema:
employees(id, name, age, salary, department_id)
departments(id, department_name)

Convert the user's question into exactly one SQLite SELECT query.
Use JOIN when department names are needed.
Return only SQL, without markdown or explanation.
Never generate modifying queries.

Question: {question}
"""

        sql = ask_groq(prompt) #ask_groq(prompt) sends the prompt to the Groq API.

        # Remove SQL markdown formatting
        sql = re.sub( #Finds and replaces matching text
            r"```(?:sql)?|```", #Matches Markdown code fences, with or without the sql label
            "",
            sql,
            flags=re.IGNORECASE #Ignores uppercase/lowercase differences.
        ).strip() #This removes unwanted Markdown formatting from the AI's reply

        # Remove optional SQL prefix
        sql = re.sub(
            r"^SQL\s*:\s*",
            "",
            sql,
            flags=re.IGNORECASE
        ).strip()
# ^ → beginning of the text.
# SQL → matches the word SQL.
# \s* → matches zero or more spaces.
# : → matches the colon.
# "" → replaces the matched prefix with nothing.
# .strip() → removes extra spaces at the beginning and end.
        return sql

    # Demo mode: no API key required
    q = question.lower()

    if "average salary" in q or "average of salary" in q:
        return """
        SELECT AVG(salary) AS average_salary
        FROM employees
        """

    if "highest salary" in q or "highest paid" in q:
        return """
        SELECT name, salary
        FROM employees
        ORDER BY salary DESC
        LIMIT 1
        """

    if "above" in q and "50000" in q:
        return """
        SELECT name, salary
        FROM employees
        WHERE salary > 50000
        """

    if "department" in q and (
        "name" in q or "each" in q
    ):
        return """
        SELECT d.department_name, e.name, e.salary
        FROM employees e
        JOIN departments d
        ON e.department_id = d.id
        """

    if "count" in q and "employee" in q:
        return """
        SELECT COUNT(*) AS employee_count
        FROM employees
        """

    if "all employees" in q or "show employees" in q:
        return "SELECT * FROM employees"

    return None


# 5. Validate and execute read-only SQL
def execute_sql(sql):

    sql = sql.strip()

    # Only SELECT or WITH queries are accepted
    if not re.match(r"^(SELECT|WITH)\b", sql, re.I): #if the query does not start with select or with then gve error
        raise ValueError("Only SELECT queries are allowed.")
#WITH query when a question requires multiple steps  give me eg of it-WITH query when a question requires multiple steps  give me eg of it
# WITH average_salary AS (
#     SELECT AVG(salary) AS avg_sal
#     FROM employees
# )
# SELECT name, salary
# FROM employees
# WHERE salary > (
#     SELECT avg_sal FROM average_salary
# );
    # Block modifying operations
    forbidden = (
        r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|"
        r"REPLACE|ATTACH|DETACH|PRAGMA)\b"
    )

    if re.search(forbidden, sql, re.I):
        raise ValueError("Unsafe SQL detected.")
#forbidden-means in the question when user uses that words then it gives error likeUnsafe SQL detected because ai may misunderstands the question and generate wrong output
# Accidental modification
# User asks: "Show Ravi's salary."
# The AI should generate a SELECT query, but suppose it generates:
# UPDATE employees SET salary = 0;
# If executed, every employee's salary becomes zero. so that this is done for security purpose of databases
# You can enforce read-only access in two main ways:
# - Read-only connection: Open the database using SQLite's read-only mode.
# - SQLite authorizer: Allow only permitted database operations, such as reading tables and running SELECT queries.
    # SQLite read-only authorizer
    allowed = {
        sqlite3.SQLITE_SELECT,
        sqlite3.SQLITE_READ,
        sqlite3.SQLITE_FUNCTION,
        getattr(sqlite3, "SQLITE_RECURSIVE", -999)
    }
# Code	Meaning
# allowed = { }	Creates a set of allowed operations.
# sqlite3.SQLITE_SELECT	Allows reading results using SELECT queries.
# sqlite3.SQLITE_READ	Allows reading data from database tables.
# sqlite3.SQLITE_FUNCTION	Allows SQL functions such as COUNT(), SUM(), and AVG().
#getattr() gets a value by its name, and the third argument is the default value if that name is missing.
    def authorizer(action, arg1, arg2, db_name, trigger):
        if action in allowed:
            return sqlite3.SQLITE_OK
        return sqlite3.SQLITE_DENY

    conn.set_authorizer(authorizer)

    try:
        cursor = conn.execute(sql)

        if cursor.description is None:
            raise ValueError("Query did not return a result.")

        columns = [
            item[0] for item in cursor.description
        ]

        # Fetch at most 101 rows to enforce the limit
        rows = cursor.fetchmany(101)

        if len(rows) > 100:
            raise ValueError(
                "Too many rows. Add a LIMIT clause."
            )

        return columns, rows

    finally:
        conn.set_authorizer(None)


# 6. Explain results in simple English
def explain_result(question, columns, rows):

    if not rows:
        return "No matching records were found."

    if api_key:
        prompt = f"""
Explain the SQL result in simple English.
Do not invent facts.

Question: {question}
Columns: {columns}
Rows: {rows[:20]}
"""

        try:
            return ask_groq(prompt)
        except Exception as error:
            print("AI explanation unavailable:", error)

    return (
        f"The query returned {len(rows)} row(s). "
        f"Result columns: {', '.join(columns)}."
    )


# 7. Run the AI SQL agent
def sql_agent(question):

    print("\n" + "=" * 50)
    print("User question:", question)

    try:
        sql = generate_sql(question)

        if not sql:
            print("Demo mode does not support this question.")
            return

        print("\nGenerated SQL:\n", sql)

        columns, rows = execute_sql(sql)

        print("\nResult:")

        print(" | ".join(columns))
        print("-" * 50)

        for row in rows:
            print(" | ".join(
                str(value) for value in row
            ))

        print("\nExplanation:")
        print(explain_result(question, columns, rows))

    except Exception as error:
        print("\nCould not execute query:", error)


# 8. Test the project
sql_agent("Show employees whose salary is above 50000")

sql_agent("Find the employee with the highest salary")

sql_agent("What is the average salary?")

# Command	Meaning
# INSERT	Add new data
# UPDATE	Modify existing data
# DELETE	Delete rows
# DROP	Delete a table or database object
# ALTER	Change a table's structure
# CREATE	Create a table or other database object
# REPLACE	Insert or replace data
# ATTACH	Attach another database
# DETACH	Disconnect an attached database
# PRAGMA	Access or change SQLite settings

Enter Groq API key (press Enter for demo mode): ··········

User question: Show employees whose salary is above 50000

Generated SQL:
 SELECT * FROM employees WHERE salary > 50000;

Result:
id | name | age | salary | department_id
--------------------------------------------------
1 | Ravi | 25 | 60000.0 | 1
2 | Priya | 28 | 75000.0 | 1
4 | Sneha | 26 | 55000.0 | 3

Explanation:
The query returned a list of employees who earn more than $50,000.  
For each employee it shows:

| id | name  | age | salary | department_id |
|----|-------|-----|--------|---------------|
| 1  | Ravi  | 25  | 60,000 | 1             |
| 2  | Priya | 28  | 75,000 | 1             |
| 4  | Sneha | 26  | 55,000 | 3             |

So Ravi, Priya, and Sneha are the only employees in the data set whose salaries exceed $50,000.

User question: Find the employee with the highest salary

Generated SQL:
 SELECT * FROM employees ORDER BY salary DESC LIMIT 1;

Result:
id | name | age | salary | department_id
--------------